# Chapter 9: Intro to Activation Functions — Modeling Probabilities
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Probability is the very guide of life."*  
> — Bishop Joseph Butler

---

## 1. Executive Summary & Objective

In earlier chapters, our output layers produced unconstrained linear values ($-\infty$ to $+\infty$), measured against Mean Squared Error. However, the majority of real-world deep learning tasks involve decision-making under uncertainty: **probability modeling**.

### Core Objectives:
1. **Enforce Kolmogorov Axioms**: Constrain network outputs to valid probability intervals ($[0, 1]$ and $\sum p_i = 1$).
2. **Deconstruct Sigmoid and Tanh**: Formulate their non-linear mappings, evaluate their analytical derivatives, and analyze the **Vanishing Gradient Problem**.
3. **Master Softmax and Cross-Entropy**: Formulate the multi-class normalized exponential operator and derive its gradient with respect to pre-activation logits.
4. **Build a Probability-Driven Classifier**: Train a neural network using Softmax and Cross-Entropy on categorical data from scratch.

---
## 2. Theoretical Deep-Dive

### 2.1 The Sigmoid Activation Function

For binary classification, we require outputs bounded strictly within $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

```
  sigma(z)
    1.0 |                    .---------
        |                   / 
    0.5 |         ---------+---------
        |                 /  
    0.0 |       ---------'   
        +--------------------+---------> z
               -4.0         0.0       +4.0
```

#### The Analytical Derivative of Sigmoid:
$$\sigma'(z) = \frac{d}{dz} (1 + e^{-z})^{-1} = -(1 + e^{-z})^{-2} (-e^{-z}) = \frac{e^{-z}}{(1 + e^{-z})^2}$$
$$\sigma'(z) = \left(\frac{1}{1 + e^{-z}}\right) \left(\frac{e^{-z}}{1 + e^{-z}}\right) = \sigma(z)(1 - \sigma(z))$$

#### The Vanishing Gradient Problem:
Notice that the maximum value of $\sigma'(z)$ is only **$0.25$** (at $z = 0$). When $z > 4$ or $z < -4$, $\sigma'(z) \to 0$. In deep networks with many sigmoid layers, multiplying numbers $< 0.25$ through the Chain Rule causes backpropagated gradients to vanish to zero before reaching early layers!

### 2.2 Softmax and Categorical Cross-Entropy

For multi-class classification among $C$ mutually exclusive classes, the **Softmax** function maps unnormalized logits $\mathbf{z} = [z_1, \dots, z_C]$ to a valid probability distribution:

$$\hat{y}_i = \text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^C e^{z_j}}$$

Properties:
1. $\hat{y}_i \in (0, 1)$ for all $i$.
2. $\sum_{i=1}^C \hat{y}_i = 1.0$.

#### Categorical Cross-Entropy Loss:
Measuring error as squared difference on probabilities is sub-optimal because MSE is non-convex when combined with sigmoid/softmax. Instead, we use the negative log-likelihood (Cross-Entropy):

$$L_{CE} = - \sum_{i=1}^C y_i \ln(\hat{y}_i)$$

When target $\mathbf{y}$ is one-hot encoded (where only true class $k$ has $y_k = 1$), this reduces simply to:
$$L_{CE} = - \ln(\hat{y}_k)$$

#### The Gradient of Softmax + Cross-Entropy:
Differentiating $L_{CE}$ with respect to the input logit $z_i$ produces an algebraic result:

$$\frac{\partial L_{CE}}{\partial z_i} = \hat{y}_i - y_i$$

The delta passed back into the network is simply **$(\text{Predicted Probability} - \text{Ground Truth Target})$**!

---
## 3. Code Reproduction

We now implement numerically stable activation functions and train a probability classifier from scratch.

In [ ]:
# 3.1 Implementing Sigmoid, Tanh, and Softmax with Numerical Stability
import numpy as np

def sigmoid(x):
    # Clamped to prevent numerical overflow in exp(-x)
    x_clamped = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x_clamped))

def sigmoid2deriv(output):
    return output * (1.0 - output)

def softmax(z):
    # Numerical stability trick: subtract max(z) along axis to prevent exp overflow
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# Demonstration
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print("Input Logits:          ", logits)
print("Softmax Probabilities: ", np.round(probs, 4))
print(f"Sum of Probabilities:   {np.sum(probs):.4f} (Must equal 1.0)")

In [ ]:
# 3.2 Cross-Entropy Loss Implementation

def cross_entropy_loss(preds, targets, eps=1e-15):
    # eps prevents taking log(0)
    preds_clipped = np.clip(preds, eps, 1.0 - eps)
    return -np.sum(targets * np.log(preds_clipped)) / len(preds)

targets_example = np.array([1.0, 0.0, 0.0])  # Class 0 is true
loss_confident = cross_entropy_loss(np.array([0.9, 0.05, 0.05]), targets_example)
loss_uncertain = cross_entropy_loss(np.array([0.34, 0.33, 0.33]), targets_example)
loss_wrong = cross_entropy_loss(np.array([0.01, 0.89, 0.10]), targets_example)

print(f"Loss (Confident & Correct): {loss_confident:.4f}")
print(f"Loss (Uncertain):           {loss_uncertain:.4f}")
print(f"Loss (Confident & Wrong):   {loss_wrong:.4f} (Heavily penalized!)")

In [ ]:
# 3.3 Training a Multi-Class Neural Network with Softmax & Cross-Entropy

np.random.seed(42)

# 3 Classes synthetic dataset (e.g., Sentiment Analysis: Positive, Neutral, Negative)
N_per_class = 150
D = 8
num_classes = 3

X0 = np.random.randn(N_per_class, D) + np.array([2.0, 0, 0, 0, 0, 0, 0, 0])
X1 = np.random.randn(N_per_class, D) + np.array([0, 2.0, 0, 0, 0, 0, 0, 0])
X2 = np.random.randn(N_per_class, D) + np.array([0, 0, 2.0, 0, 0, 0, 0, 0])
X = np.vstack([X0, X1, X2])

y = np.zeros((len(X), num_classes))
y[:N_per_class, 0] = 1.0
y[N_per_class:2*N_per_class, 1] = 1.0
y[2*N_per_class:, 2] = 1.0

# Shuffle
indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

# Architecture
hidden_dim = 16
W1 = 0.1 * np.random.randn(D, hidden_dim)
b1 = np.zeros((1, hidden_dim))
W2 = 0.1 * np.random.randn(hidden_dim, num_classes)
b2 = np.zeros((1, num_classes))

alpha = 0.02
epochs = 250

for epoch in range(epochs):
    # 1. Forward Pass
    z1 = np.dot(X, W1) + b1
    a1 = np.tanh(z1)                     # Tanh hidden activation
    z2 = np.dot(a1, W2) + b2
    probs = softmax(z2)                  # Softmax output
    
    loss = cross_entropy_loss(probs, y)
    accuracy = np.mean(np.argmax(probs, axis=1) == np.argmax(y, axis=1)) * 100
    
    # 2. Backward Pass
    delta2 = (probs - y) / len(X)        # Output delta
    dW2 = np.dot(a1.T, delta2)
    db2 = np.sum(delta2, axis=0, keepdims=True)
    
    delta1 = np.dot(delta2, W2.T) * (1.0 - a1 ** 2)  # Tanh derivative: 1 - tanh^2
    dW1 = np.dot(X.T, delta1)
    db1 = np.sum(delta1, axis=0, keepdims=True)
    
    # 3. Parameter Updates
    W2 -= alpha * dW2
    b2 -= alpha * db2
    W1 -= alpha * dW1
    b1 -= alpha * db1
    
    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Cross-Entropy Loss: {loss:.4f} | Accuracy: {accuracy:.2f}%")

---
## 4. Chapter 9 Summary & Key Takeaways

| Activation Function | Formulation | Derivative | Primary Use Case |
|:---|:---:|:---:|:---|
| **Sigmoid** | $\frac{1}{1 + e^{-z}}$ | $\sigma(1 - \sigma)$ | Binary classification outputs; gating mechanism in LSTMs. |
| **Tanh** | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - \tanh^2(z)$ | Zero-centered hidden representations. |
| **Softmax** | $\frac{e^{z_i}}{\sum e^{z_j}}$ | $\hat{y}_i - y_i$ (with $L_{CE}$) | Multi-class mutually exclusive probability distributions. |
| **Cross-Entropy Loss** | $-\sum y_i \ln(\hat{y}_i)$ | Natural partner for Softmax, forming smooth convex loss without saturation. |
| **Next Step** | In **Chapter 10**, we tackle 2D spatial structures directly by constructing **Convolutional Neural Networks (CNNs)** from scratch. |